In [ ]:
from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Add the name of the project in google console
client = bigquery.Client(project="citric-trees-489611-k5")

# Data Preparation - Patient Table

In [ ]:
# Load the data
patient_query = """
SELECT  pat.subject_id,
        pat.anchor_age

FROM    physionet-data.mimiciv_3_1_hosp.patients pat
"""
# patient_query_job = client.query(patient_query)
# patient_df = patient_query_job.result().to_dataframe()
df_age = client.query(patient_query).to_dataframe()

print(df_age.info())
print(df_age.describe())
print("Nulls:\n", df_age.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364627 entries, 0 to 364626
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   subject_id  364627 non-null  Int64
 1   anchor_age  364627 non-null  Int64
dtypes: Int64(2)
memory usage: 6.3 MB
None
            subject_id  anchor_age
count         364627.0    364627.0
mean   15011664.762544   48.875097
std     2885012.874923   20.943316
min         10000032.0        18.0
25%         12512008.5        29.0
50%         15018682.0        48.0
75%         17509003.0        65.0
max         19999987.0        91.0
Nulls:
 subject_id    0
anchor_age    0
dtype: int64


In [ ]:
# Handle the 91+ anonymization artifact
## MIMIC-IV shifts ages >89 to 91 to protect patient identity.
## Two approaches:
### 1) Treat 91 as a sentinel ">=90" value (keep it, acknowledge in docs)
### 2) Clip everything at 89 to remove ambiguity
## Go with Option 1

In [ ]:
# Bin into clinical age groups
bins = [0, 18, 40, 65, 80, 200]
labels = ["<18", "18-40", "41-65", "66-80", "81+"]

df_age["age_group"] = pd.cut(
    df_age["anchor_age"],
    bins=bins,
    labels=labels,
    right=True
)

# # One-hot encode if your model needs it (tree-based models often prefer ordinal)
# age_dummies = pd.get_dummies(df_age["age_group"], prefix="age_grp")
# df_age = pd.concat([df_age, age_dummies], axis=1)

In [ ]:
# Normalize continuous age
scaler = MinMaxScaler()
df_age["anchor_age_norm"] = scaler.fit_transform(df_age[["anchor_age"]])

In [ ]:
# Final cleaned table
df_age_final = df_age[[
    "subject_id",
    "anchor_age",           # original continuous feature (91 = sentinel for ≥90)
    "anchor_age_norm",      # min-max scaled for RL input
    "age_group",            # categorical label
    *[c for c in df_age.columns if c.startswith("age_grp_")]
]]

print(df_age_final.head())
print(df_age_final.shape)

   subject_id  anchor_age  anchor_age_norm age_group
0    10078138          18              0.0       <18
1    10180372          18              0.0       <18
2    10686175          18              0.0       <18
3    10851602          18              0.0       <18
4    10902424          18              0.0       <18
(364627, 4)


In [ ]:
from huggingface_hub import login
login('')

In [ ]:
from datasets import Dataset, DatasetInfo

PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

age_info = DatasetInfo(
    description=(
        "Patient age features derived from the MIMIC-IV hosp.patients table. "
        "One row per patient (subject_id). "
        "anchor_age is retained as-is, including the MIMIC-IV sentinel value of 91 "
        "which represents patients aged 90 or older (HIPAA de-identification). "
        "Includes min-max normalized age (anchor_age_norm), a clinical age group "
        "categorical (age_group: <18, 18-40, 41-65, 66-80, 81+), and one-hot "
        "encoded age group columns for model input."
    ),
    license=PHYSIONET_LICENSE,
)

ds_age = Dataset.from_pandas(df_age_final, split='age_features', info=age_info)
ds_age.push_to_hub("ADS599-Capstone/modeling_data", config_name="age_features", data_dir="modeling_data")
print("Pushed age_features to HuggingFace Hub.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/365 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  96%|#########5| 2.67MB / 2.79MB            

README.md: 0.00B [00:00, ?B/s]

Pushed age_features to HuggingFace Hub.
